# Random Agent Baseline
Runs the random policy over N episodes and measures reward.

In [1]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
from src.env.warehouse_env import WarehouseEnv
from src.analytics import RewardTracker

with open('../configs/env_config.yaml') as f:
    config = yaml.safe_load(f)

env = WarehouseEnv(config)
print(f"Env: {config['env']['name']}  |  Agents: {env.n_agents}  |  Action dim: {env.action_dim}")

Env: rware-tiny-2ag-v2  |  Agents: 2  |  Action dim: 5


In [2]:
def random_policy(obs, action_dim, env):
    ACTION_INTERACT = 4
    actions = [np.random.randint(action_dim) for _ in obs]
    u = env._env.unwrapped
    goal_set = {(gc, gr) for gc, gr in u.goals}
    shelf_set = {(s.x, s.y) for s in u.shelfs}
    for i, agent in enumerate(u.agents):
        pos = (agent.x, agent.y)
        if agent.carrying_shelf and pos in goal_set:
            actions[i] = ACTION_INTERACT
        elif agent.carrying_shelf:
            actions[i] = np.random.randint(4)
        elif not agent.carrying_shelf and pos in goal_set:
            actions[i] = np.random.randint(4)
        elif not agent.carrying_shelf and pos in shelf_set:
            actions[i] = ACTION_INTERACT
    return actions

In [3]:
N_EPISODES = 10000
max_steps = config['env'].get('max_steps', 500)
tracker = RewardTracker(n_agents=env.n_agents)

for ep in range(N_EPISODES):
    obs = env.reset()
    tracker.start_episode()
    for _ in range(max_steps):
        actions = random_policy(obs, env.action_dim, env)
        obs, rews, dones, _ = env.step(actions)
        tracker.record_step(rews)
        if all(dones):
            break
    tracker.end_episode()
    ep_data = tracker.episodes[-1]
    print(f"  ep {ep+1:3d}/{N_EPISODES}  steps={ep_data['n_steps']:4d}  team_reward={ep_data['team_total_reward']:.3f}")

env.close()

/Users/jennykim/Documents/CS5180/smart-warehouse/.venv/lib/python3.14/site-packages/gymnasium/utils/passive_env_checker.py:245: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'list'>
  logger.warn(


  ep   1/10000  steps= 500  team_reward=0.000
  ep   2/10000  steps= 500  team_reward=0.000
  ep   3/10000  steps= 500  team_reward=0.000
  ep   4/10000  steps= 500  team_reward=0.000
  ep   5/10000  steps= 500  team_reward=0.000
  ep   6/10000  steps= 500  team_reward=0.000
  ep   7/10000  steps= 500  team_reward=0.000
  ep   8/10000  steps= 500  team_reward=0.000
  ep   9/10000  steps= 500  team_reward=0.000
  ep  10/10000  steps= 500  team_reward=0.000
  ep  11/10000  steps= 500  team_reward=0.000
  ep  12/10000  steps= 500  team_reward=0.000
  ep  13/10000  steps= 500  team_reward=0.000
  ep  14/10000  steps= 500  team_reward=0.000
  ep  15/10000  steps= 500  team_reward=0.000
  ep  16/10000  steps= 500  team_reward=0.000
  ep  17/10000  steps= 500  team_reward=0.000
  ep  18/10000  steps= 500  team_reward=0.000
  ep  19/10000  steps= 500  team_reward=0.000
  ep  20/10000  steps= 500  team_reward=0.000
  ep  21/10000  steps= 500  team_reward=0.000
  ep  22/10000  steps= 500  team_r

In [4]:
summary = tracker.summary()
print('--- Random Baseline Summary ---')
print(f"  Episodes : {summary['n_episodes']}")
print(f"  Team total reward  — mean: {summary['team_total_reward']['mean']:.4f}  "
      f"std: {summary['team_total_reward']['std']:.4f}  "
      f"[{summary['team_total_reward']['min']:.4f}, {summary['team_total_reward']['max']:.4f}]")
print(f"  Episode length     — mean: {summary['episode_length']['mean']:.1f}")
for i, v in enumerate(summary['per_agent_mean_total']):
    print(f"  Agent {i} mean total reward: {v:.4f}")

--- Random Baseline Summary ---
  Episodes : 10000
  Team total reward  — mean: 0.0658  std: 0.2535  [0.0000, 3.0000]
  Episode length     — mean: 500.0
  Agent 0 mean total reward: 0.0342
  Agent 1 mean total reward: 0.0316


In [5]:
import matplotlib.pyplot as plt

rewards = [ep['team_total_reward'] for ep in tracker.episodes]
plt.figure(figsize=(10, 4))
plt.bar(range(len(rewards)), rewards, color='steelblue', alpha=0.7)
plt.axhline(summary['team_total_reward']['mean'], color='red', linestyle='--', label=f"mean={summary['team_total_reward']['mean']:.3f}")
plt.xlabel('Episode')
plt.ylabel('Team Total Reward')
plt.title('Random Agent Baseline — Reward per Episode')
plt.legend()
plt.tight_layout()
plt.savefig('../results/plots/random_baseline_reward.png', dpi=150)
plt.show()

/var/folders/gb/c0f1t3sn061g3305zd78xrj40000gn/T/ipykernel_51108/366180730.py:12: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig('../results/plots/random_baseline_reward.png', dpi=150)
/var/folders/gb/c0f1t3sn061g3305zd78xrj40000gn/T/ipykernel_51108/366180730.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
tracker.save('../results/logs/random_baseline_rewards.json')
tracker.save_csv('../results/logs/random_baseline_rewards.csv')

[analytics] Saved reward log → ../results/logs/random_baseline_rewards.json
[analytics] Saved CSV → ../results/logs/random_baseline_rewards.csv
